# 0. Libraries

In [1]:
pip install simplemma

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
import warnings

# Filter out the specific UserWarnings
warnings.filterwarnings("ignore", category=UserWarning, message="A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy")
warnings.filterwarnings("ignore", category=UserWarning, message="unable to load libtensorflow_io_plugins.so")
warnings.filterwarnings("ignore", category=UserWarning, message="file system plugins are not loaded")

In [4]:
# Accuracy metrics from Scikit-Learn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report

In [5]:
# Hugging Face library
from datasets import Dataset, DatasetDict

In [6]:
# NLP libraries
import re
import nltk
import simplemma

from simplemma import text_lemmatizer
from nltk.corpus import stopwords

# 1. Load Dataset

In [7]:
# Create a function to import the data from csv format
def load_data(file_path):
    return pd.read_csv(file_path, header=None, delimiter='\t', names=['sentiment', 'text'])


train_path = '/kaggle/input/sentiment/train_bal_vdg_27_11.tsv'
test_path = '/kaggle/input/sentiment/test_bal_vdg_27_11.tsv'
val_path = '/kaggle/input/sentiment/valid_bal_vdg_27_11.tsv'

df_train = load_data(train_path)
df_test = load_data(test_path)
df_val = load_data(val_path)

In [8]:
def converter(df): 
    mapping = {'NEG':'negative', 'NEU':'neutral', 'POS':'positive'} 
    df['sentiment'] = df['sentiment'].replace(mapping) 
    return df

df_train = converter(df_train) 
df_val = converter(df_val) 
df_test = converter(df_test)

In [9]:
# To get an idea of the data
pd.set_option('display.max_colwidth', 150)
df_train.head()

,sentiment,text
0,neutral,"La lotta contro il bodyshaming è una cosa, promuovere l’obesità è un’altra"
1,positive,"La marginalità è un luogo radicale di possibilità, uno spazio di #resistenza. Ne parla @Racheleborghi in questo podcast pubblicato da #TRANSfemmIN..."
2,neutral,"@ilgiomba @GiovannaSerra3 @voilaloves @ArthurMeurs @LuniVale @antrichelieu @diegodemme4 @fabyo255 Io spero solo,che in cuor suo,Giovanna abbia ca..."
3,positive,Seppellire l'odio sotto una montagna di amore #ProudBoys 🧡💜💙💚♥️
4,neutral,#iorestoacasama non dimentico. E SE IO LOTTO DA PARTIGIANA Raccolta delle biografie delle partigiane a cura di @NonUnaDiMenoMI https://t.co/KUlwqE...


In [10]:
# Remove user mention here. could not do it in the preprocess function
df_train['text'] = df_train['text'].str.replace('@[A-Za-z0-9]+\s?', '', regex=True)
df_val['text'] = df_val['text'].str.replace('@[A-Za-z0-9]+\s?', '', regex=True)
df_test['text'] = df_test['text'].str.replace('@[A-Za-z0-9]+\s?', '', regex=True)

In [11]:
# I'm combining the pandas dataframe to the dataset dictionary of Hugging Face

train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)
val_dataset = Dataset.from_pandas(df_val)

# Create the DatasetDict
dataset = DatasetDict({'train': train_dataset, 'test': test_dataset, 'validation': val_dataset})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['sentiment', 'text'],
        num_rows: 717
    })
    test: Dataset({
        features: ['sentiment', 'text'],
        num_rows: 216
    })
    validation: Dataset({
        features: ['sentiment', 'text'],
        num_rows: 92
    })
})


In [12]:
# Removing duplicates

# Initialize a dictionary to store updated datasets
updated_datasets = {}

# Check for and remove duplicates in each split
for split in dataset.keys():
    split_data = dataset[split]
    
    # Access the 'text' column within the list
    text_column = split_data['text']
    
    # Initialize a set to track unique texts
    unique_texts = set()
    
    # Initialize lists to store the filtered data
    filtered_text = []
    
    # Iterate through the 'text' column and filter duplicates
    for text in text_column:
        if text not in unique_texts:
            unique_texts.add(text)
            filtered_text.append(text)
    
    # Create a new Dataset object with the filtered data
    updated_datasets[split] = split_data.select(list(range(len(filtered_text))))
    
    # Print the number of removed duplicates
    duplicate_count = len(text_column) - len(filtered_text)
    print(f"Duplicates removed in {split} split: {duplicate_count}\n")

# Update the dataset dictionary with the filtered datasets
dataset.update(updated_datasets)

# Print the updated dataset information
for split in dataset.keys():
    split_data = dataset[split]
    print(f"{split}: {len(split_data['text'])} rows")

print(dataset)

Duplicates removed in train split: 2

Duplicates removed in test split: 0

Duplicates removed in validation split: 1

train: 715 rows
test: 216 rows
validation: 91 rows
DatasetDict({
    train: Dataset({
        features: ['sentiment', 'text'],
        num_rows: 715
    })
    test: Dataset({
        features: ['sentiment', 'text'],
        num_rows: 216
    })
    validation: Dataset({
        features: ['sentiment', 'text'],
        num_rows: 91
    })
})


# 2. Data Preprocessing

In [13]:
italian_stopwords = set(stopwords.words('italian'))

# Define a function to preprocess text
import re

def preprocess_text(text):    
    pattern = r'no(?=\s*[.!?])'  # Matches "no" followed by zero or more spaces and sentence-ending punctuation
    
    # Remove "no" at the end of sentences
    text = re.sub(pattern, '', text)
    # Tokenization, lemmatization, removing punctuation, stopwords, and URLs
    text = text_lemmatizer(text, lang='it')
    text = ' '.join(text)
    
    text = re.sub(r'[^\w\s\']', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Define a regular expression pattern to match "no" at the end of a sentence

    
    text = ' '.join(word for word in text.split() if word.lower() not in italian_stopwords)
    
    return text





def preprocess_dataset(dataset):
    dataset['text'] = preprocess_text(dataset['text'])
    return dataset

dataset = dataset.map(preprocess_dataset)

  0%|          | 0/715 [00:00<?, ?ex/s]

  0%|          | 0/216 [00:00<?, ?ex/s]

  0%|          | 0/91 [00:00<?, ?ex/s]

# 3. VADER 

In [14]:
import re

def remove_no_at_end_of_sentence(text):
    # Define a regular expression pattern to match "no" at the end of a sentence
    pattern = r'no(?=\s*[.!?])'  # Matches "no" followed by zero or more spaces and sentence-ending punctuation
    
    # Remove "no" at the end of sentences
    modified_text = re.sub(pattern, '', text)
    
    return modified_text

# Example usage:
text = "This is a sentence with no."
modified_text = remove_no_at_end_of_sentence(text)
print(modified_text)



This is a sentence with .


In [15]:
# nltk.download('vader_lexicon')

In [16]:
pip install vader-multi

Note: you may need to restart the kernel to use updated packages.


In [17]:
# Import the lexicon 
# This version integrates the Google Translate API through the translatte Python library. 
# It requires an active Internet connection in order to work. Text language is automatically detected so it behaves exactly like the original version

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

In [18]:
# Analyze Italian text
italian_text = dataset['train']['text'][94]
sentiment_scores = sia.polarity_scores(italian_text)

# The sentiment_scores will contain the sentiment polarity (positive, negative, neutral) and a compound score.
print(sentiment_scores)

{'neg': 0.551, 'neu': 0.449, 'pos': 0.0, 'compound': -0.8625}


In [19]:
def format_output(output_dict):
    polarity = "neutral"
    
    if(output_dict['compound']>= 0.05):
        polarity = "positive"
    
    elif(output_dict['compound']<= -0.05):
        polarity = "negative"
    
    return polarity

In [20]:
def vader(dataset):
    dataset['vader_prediction'] = sia.polarity_scores(dataset['text'])
    dataset['vader_prediction'] = format_output(dataset['vader_prediction'])
    return dataset

dataset = dataset.map(vader)

  0%|          | 0/715 [00:00<?, ?ex/s]

  0%|          | 0/216 [00:00<?, ?ex/s]

  0%|          | 0/91 [00:00<?, ?ex/s]

# 4. Metrics

In [21]:
predicted_labels = dataset['test']['vader_prediction']
label_test = dataset['test']['sentiment']

In [22]:
print(classification_report(label_test,predicted_labels))

              precision    recall  f1-score   support

    negative       0.45      0.69      0.55        80
     neutral       0.50      0.14      0.22        63
    positive       0.55      0.58      0.56        73

    accuracy                           0.49       216
   macro avg       0.50      0.47      0.44       216
weighted avg       0.50      0.49      0.46       216



In [23]:
accuracy = accuracy_score(label_test, predicted_labels) # (TP+TN)/P+N i.e total number of corrected classified tweet over total number of tweets

print(accuracy)

0.49074074074074076


In [24]:
precision = precision_score(label_test, predicted_labels,average=None, labels=['negative','neutral','positive']) # TP/(TP+FP) i.e if predicted a certain class, which is the probability of being really that class?

print(precision)

[0.45454545 0.5        0.54545455]


In [25]:
recall = recall_score(label_test, predicted_labels,average=None, labels=['negative','neutral','positive']) # TP/(TP+FN) i.e the ability of the estimator to predict all the tweets of a given class

print(recall)

[0.6875     0.14285714 0.57534247]


In [26]:
f1score = f1_score(label_test, predicted_labels,average=None, labels=['negative','neutral','positive']) # 2*(precision*recall)/(precision+recall)

print(f1score)

[0.54726368 0.22222222 0.56      ]
